# FLP - příprava na cvičení č. 6 (Rust 3) - až 2b
Toto je notebook s přípravou na výše uvedené cvičení.

Pokyny:
- Nastudujte si informace uvedené v tomto notebooku.
- Vyřešte příklady uvedené v sekcích **PŘÍKLAD**, např. doplněním zdrojového kódu do vyznačených částí / opravou kódu - dle pokynů.
- Do kódu/textu mimo příklady **nezasahujte**, žádné další buňky/bloky kódu **nepřidávejte**.
- Notebook s vyřešenými příklady nahrajte do Moodlu v sekci **Odevzdání domácích příprav**.

Na cvičení se bude **předpokládat** znalost zde uvedené problematiky, stejně jako znalost látky z dosud proběhlých přednášek!

Je **PŘÍSNĚ ZAKÁZÁNO** tento soubor poskytovat jiným osobám, nebo s nimi sdílet své řešení!

### Poznámka k využití generativní AI při řešení
Využití generativní AI zakázáno není, avšak **silně doporučujeme** se nejprve nad příkladem zamyslet **samostatně** a konzultaci s generativní AI brát až jako poslední možnost. Tento notebook je zde pro Vás, abyste se něco naučili, něco si vyzkoušeli a trochu se u toho "potrápili". Získané zkušenosti se vám budou hodit na **cvičeních**, při řešení **projektu** i u **zkoušky**. Necháte-li si "řešení vygenerovat chatbotem", ochuzujete především sami sebe.

# Vícemodulové programy

## Crate
- **Crate** je *kompilační jednotka*.
- V Cargo projektu máme:
  - **bin crate** (spustitelný program) – vstupní soubor je obvykle `src/main.rs`
  - **lib crate** (knihovna) – vstupní soubor je obvykle `src/lib.rs`
- Jedno Cargo "package" (adresář s `Cargo.toml`) může obsahovat:
  - **jednu knihovnu** (`src/lib.rs`)
  - **libovolně binárek** (např. `src/main.rs` + další v `src/bin/*.rs`)
- Každý crate má svůj "kořen": `crate::...` začíná od root modulu (`main.rs` nebo `lib.rs`).

## Module (modul)
- **Modul** je *jmenovaný namespace* uvnitř crate.
- Modul může být:
  - v samostatném souboru (`mod x;` + `x.rs` / `x/mod.rs`)
  - inline (`mod x { ... }`)
- Viditelnost:
  - `pub` = položka je veřejná a lze k ní přistupovat i z jiných modulů, pokud je veřejná i cesta k ní,
  - bez `pub` = položka je privátní - je přístupná pro aktuální modul a jeho potomky.

## Ukázka vícemodulového programu
Mějme Cargo projekt se spustitelným programem (binary crate):
```text
my_app/
├── Cargo.toml
└── src/
    ├── main.rs
    ├── math.rs
    └── utils/
        ├── mod.rs
        └── text.rs
```

### `src/main.rs`
```rust
mod math;
mod utils;

use math::add;
use utils::greet;
use utils::text::shout;

fn main() {
    let x = add(2, 3);
    println!("2 + 3 = {}", x);

    greet("Honzo");

    let msg = shout("ahoj svete");
    println!("{}", msg);
}
```

### `src/math.rs`
```rust
pub fn add(a: i32, b: i32) -> i32 {
    a + b
}
```

### `src/utils/mod.rs`
```rust
pub mod text;

pub fn greet(name: &str) {
    println!("Ahoj, {}!", name);
}
```

### `src/utils/text.rs`
```rust
pub fn shout(s: &str) -> String {
    s.to_uppercase()
}
```

# Iterátory - opakování a pokročilejší příklady

**Iterátor** postupně vrací prvky pomocí metody `next()`.

Můžeme jej vytvořit:
- z kolekce/kontejneru: `iter()`, `iter_mut()`, `into_iter()`
- rozsahem: `0..10`, `1..=5`
- z řetězce: `split_whitespace()`, `lines()`, `chars()`, ...

## Tvorba iterátoru z kolekce

### `iter()`
- Vrací **immutable reference** (jen pro čtení) na prvky (`&T`).
- Použijeme, když chceme data jen **číst**
- Původní kolekce se **nespotřebuje**, po iteraci ji můžeme dále používat.

In [3]:
let v = vec![10, 20, 30];

for x in v.iter() {
    println!("prvek = {}", x);
}

println!("puvodni vektor = {:?}", v); // stale lze pouzit

prvek = 10
prvek = 20
prvek = 30
puvodni vektor = [10, 20, 30]


### `iter_mut()`
- Vrací **mutable reference** na prvky (`&mut T`).
- Použijeme, když chceme prvky **měnit na místě**.
- Původní kolekce se **nespotřebuje**, po iteraci ji můžeme dále používat.

In [4]:
let mut v = vec![10, 20, 30];

for x in v.iter_mut() {
    *x += 1;
}

println!("upraveny vektor = {:?}", v); // [11, 21, 31]

upraveny vektor = [11, 21, 31]


### `into_iter()`
- Vrací přímo prvky (`T`).
- Použijeme, když chceme **převzít vlastnictví** prvků.
- Kolekci **spotřebuje** a už nejde dále použít.

In [5]:
let v = vec![10, 20, 30];

for x in v.into_iter() {
    println!("prvek = {}", x);
}

prvek = 10
prvek = 20
prvek = 30


()

## Tvorba iterátoru z řetězce

In [6]:
let text = "ahoj svete z rustu";

let words = text.split_whitespace(); // iterator pres slova

for w in words {
    println!("{}", w);
}

ahoj
svete
z
rustu


()

## Transformace (adaptéry)
- Metody, které z existujícího iterátoru vytvoří **nový iterátor**.
- Většina je **líná** - samy nic nepočítají, jen popisují, **jak se mají prvky zpracovat**, až je někdo začne skutečně číst.
- Lze je **spojovat** do složitějších operací.

### Přehled nejčastějších transformací

- `map(f)` - převede každý prvek na jinou hodnotu,
- `filter(podmínka)` - ponechá jen prvky splňující podmínku,
- `take(n)` - vezme prvních `n` prvků,
- `skip(n)` - přeskočí prvních `n` prvků,
- `enumerate()` - přidá ke každému prvku jeho index,
- `zip(it2)` - spojí dva iterátory do dvojic,
- `rev()` - obrátí pořadí iterace,
- `copied()` - z `&T` udělá `T` pro `Copy` typy,
- `cloned()` - z `&T` udělá `T` pomocí `Clone`,
- `filter_map(f)` - transformace + odfiltrování neplatných výsledků,
- `flat_map(f)` - převod na více prvků + zploštění,
- `chain(it2)` - spojí dva iterátory za sebe,
- `inspect(f)` - dovolí ladicí výpis bez změny prvků.

### Spojování transformací

In [7]:
let a = [1, 2, 3, 4, 5, 6, 7, 8];

let out: Vec<i32> = a.iter()
    .copied()               // &i32 -> i32
    .skip(2)                // preskoci 1, 2
    .take(4)                // vezme 3, 4, 5, 6
    .filter(|x| x % 2 == 0) // necha jen suda cisla
    .map(|x| x * x)         // umocni
    .collect();             // ulozi do Vec

println!("{:?}", out);      // [16, 36]

[16, 36]


## Spotřebující (consuming) metody
- Čtou prvky a provádějí určitý výpočet nebo akci.
- Iterátor skutečně **projdou** a **spotřebují**.
- Po dokončení už původní iterátor obvykle nelze znovu použít.

### Přehled nejčastějších spotřebujících metod

- `collect()` - projde všechny prvky a uloží je do kolekce, např. `Vec<T>`,
- `sum()` - sečte všechny prvky iterátoru,
- `count()` - spočítá počet prvků,
- `find(podmínka)` - vrátí první prvek splňující podmínku - vrací `Option<...>`,
- `fold(počáteční_hodnota, funkce)` - obecná akumulace prvků do jedné výsledné hodnoty,
- `for_each(f)` - provede zadanou akci pro každý prvek,
- `any(podmínka)` - vrátí `true`, pokud alespoň jeden prvek splňuje podmínku,
- `all(podmínka)` - vrátí `true`, pokud všechny prvky splňují podmínku.

### Ukázka použití

In [8]:
let a = [1, 2, 3, 4, 5, 6];

let sum_even_squares: i32 = a.iter()
    .copied()
    .filter(|x| x % 2 == 0)
    .map(|x| x * x)
    .sum();

println!("{}", sum_even_squares); // 56

56


### **PŘÍKLAD 1:** Pokročilá práce s iterátory
Pomocí iterátorů, adaptérů a spotřebujících metod doplňte implementaci funkcí:
- `sum_doubled_even_lt` - Vybere sudá čísla menší než `threshold`, zdvojnásobí je a vrátí jejich součet.
- `first_divisible_label` - Najde první prvek dělitelný `k`.
  - Pokud existuje, vrátí `Some` ve formátu `"index:hodnota"`.
  - Pokud neexistuje, vrátí `None`.
- `last_gt` - Vrátí poslední číslo větší než `k`.
  - Pokud existuje, vrátí `Some(hodnota)`.
  - Pokud neexistuje, vrátí `None`.
- `rising_pair_labels` - Najde všechny sousední dvojice čísel `a`,`b`, kde druhé číslo (`b`) je větší než první (`a`). Výsledek vrátí jako vektor řetězců `"index:a->b"`.
- Do funkce `main` a definice názvů/parametrů/návratových hodnot funkcí **nezasahujte**!

In [9]:
fn sum_doubled_even_lt(v: &[i32], threshold: i32) -> i32 {
    // Ze vstupniho pole/slice:
    // 1) vyberte jen suda cisla mensi nez threshold
    // 2) kazde vynasobte 2
    // 3) vratte jejich soucet

    v.iter().filter(|&&x| x % 2 == 0 && x < threshold).map(|&x| x * 2).sum()
}

fn first_divisible_label(v: &[i32], k: i32) -> Option<String> {
    // Najdete prvni prvek delitelny k.
    // Pokud existuje, vratte Some ve formatu "index:hodnota".
    // Pokud neexistuje, vratte None.
    //
    // Priklad:
    // v = [5, 7, 12, 9], k = 3
    // -> Some("2:12")

    v.iter().enumerate().find(|&(_, &x)| x % k == 0).map(|(i, &x)| format!("{}:{}", i, x))
}


// Funkce `last_gt`:
// Najde posledni prvek vetsi nez `k`.
// Pokud existuje, vrati `Some(hodnota)`.
// Pokud neexistuje, vrati `None`.
//
// Priklad:
// v = [5, 12, 3, 8, 15, 2], k = 10
// -> Some(15)
fn last_gt(v: &[i32], k: i32) -> Option<i32> {
    v.iter().copied().filter(|&x| x > k).last()
}


// Funkce `rising_pair_labels`:
// Najde vsechny sousedni dvojice, ve kterych druhe cislo je vetsi nez prvni.
// Vrati je jako `Vec<String>` ve formatu `"index:a->b"`,
// kde `index` je index prvniho prvku dane dvojice.
//
// Priklad:
// v = [5, 12, 3, 8, 15, 2, 7]
// sousedni dvojice:
//   (5,12)  -> roste  -> "0:5->12"
//   (12,3)  -> neroste
//   (3,8)   -> roste  -> "2:3->8"
//   (8,15)  -> roste  -> "3:8->15"
//   (15,2)  -> neroste
//   (2,7)   -> roste  -> "5:2->7"
//
// Vysledek:
// vec!["0:5->12", "2:3->8", "3:8->15", "5:2->7"]
fn rising_pair_labels(v: &[i32]) -> Vec<String> {
    v.windows(2).enumerate().filter(|&(_, pair)| pair[1] > pair[0]).map(|(i, pair)| format!("{}:{}->{}", i, pair[0], pair[1])).collect()
}

fn main() {
    let v = vec![5, 12, 3, 8, 15, 2, 7, 20, 9];
    
    let s = sum_doubled_even_lt(&v, 15);
    println!("sum_doubled_even_lt(15) = {}", s);
    assert_eq!(s, 24 + 16 + 4); // 12*2 + 8*2 + 2*2
    
    let f1 = first_divisible_label(&v, 3);
    println!("first_divisible_label(3) = {:?}", f1);
    assert_eq!(f1, Some(String::from("1:12")));
    
    let f2 = first_divisible_label(&v, 11);
    println!("first_divisible_label(11) = {:?}", f2);
    assert_eq!(f2, None);

    let l1 = last_gt(&v, 10);
    println!("last_gt(10) = {:?}", l1);
    assert_eq!(l1, Some(20));

    let l2 = last_gt(&v, 100);
    println!("last_gt(100) = {:?}", l2);
    assert_eq!(l2, None);

    let pairs = rising_pair_labels(&v);
    println!("rising_pair_labels() = {:?}", pairs);
    assert_eq!(
        pairs,
        vec![
            String::from("0:5->12"),
            String::from("2:3->8"),
            String::from("3:8->15"),
            String::from("5:2->7"),
            String::from("6:7->20"),
        ]
    );
}
main();

sum_doubled_even_lt(15) = 44
first_divisible_label(3) = Some("1:12")
first_divisible_label(11) = None
last_gt(10) = Some(20)
last_gt(100) = None
rising_pair_labels() = ["0:5->12", "2:3->8", "3:8->15", "5:2->7", "6:7->20"]


# Vracení hodnot a referencí funkcemi

- Když funkce **vrací hodnotu**, vrací samostatný výsledek, který **není odkazem do vstupních dat**.
- Když funkce **vrací referenci**, nevrací nový obsah, ale jen **odkaz na data, která už existují jinde**.

To znamená, že:
- U návratové reference je důležité, **na která data ukazuje**.
- A v Rustu zejména, **zda tato data budou ještě existovat**.

## Funkce vrací obyčejnou hodnotu

```rust
fn len_of(s: &str) -> usize {
    s.len()
}

fn main() {
    let text = String::from("ahoj");
    let n = len_of(&text);
    println!("delka = {n}");
}
```

Tato funkce vrací `usize`, tedy novou hodnotu.
Návratová hodnota není reference do `text`, takže po návratu funkce není potřeba řešit, na co ukazuje.

## Funkce vrací referenci

```rust
fn identity(s: &str) -> &str {
    s
}

fn main() {
    let text = String::from("ahoj");
    let out = identity(&text);
    println!("{out}");
}
```

Tady funkce nevrací novou hodnotu, ale referenci na existující data.
Vrácená reference tedy "nějak souvisí se vstupem".

Říkáme, že:
- Lifetime návratové reference je **svázaný s lifetime vstupní reference**.
- Jinými slovy: vrácenou referenci lze používat jen tak dlouho, **dokud žijí data, na která ukazuje.**

## Proč je vrácení reference citlivější?

- U návratu `usize` je jedno, co se po skončení funkce stane se vstupní referencí.
- U návratu `&str` to jedno není - vrácená reference **musí ukazovat na data, která jsou stále platná**.

## Jak číst signatury funkcí?

```rust
fn f1(x: String)         // Funkce přebírá hodnotu.
fn f2(x: &str)           // Funkce si hodnotu půjčuje pro čtení.
fn f3(x: &mut String)    // Funkce si hodnotu půjčuje pro změnu.
fn f4(x: &str) -> usize  // Funkce vrací novou hodnotu.
fn f5(x: &str) -> &str   // Funkce vrací referenci navázanou na existující data.
```

Podle signatury jde tedy poznat, zda si funkce hodnotu přebere, pouze si ji výpůjčí a zda ji může měnit. Dále lze také identifikovat, jestli vrací referenci do existujících dat.

### **PŘÍKLAD 2:** Doplňování signatur funkcí
- **Doplňte signatury** (konkrétně parametry + návratové hodnoty) funkcí v níže uvedeném programu.
- Do těla funkcí, funkce main, ani jinam nezasahujte.

In [11]:
// Funkce `command_name`:
// Vrátí název příkazu, tedy část textu před prvním znakem ':'.
//
// Příklad volání:
// - command_name("LOGIN:alice!") -> "LOGIN"
// - command_name("PING:server42") -> "PING"
//
// Funkce si vstupní text pouze půjčuje pro čtení.
// Nevrací nový String, ale jen pohled do původního textu.
fn command_name(
    cmd: &str
) -> &str
{
    cmd.split(':').next().unwrap_or("")
}

// Funkce `payload`:
// Vrátí payload, tedy část textu za prvním znakem ':'.
//
// Příklad volání:
// - payload("LOGIN:alice!") -> "alice!"
// - payload("PING:server42") -> "server42"
//
// Pokud ':' ve vstupu chybí, vrátí prázdný řetězec "".
//
// Funkce si vstupní text pouze půjčuje pro čtení.
// Výstupem je reference do původního textu.
fn payload(
    cmd: &str
)
-> &str
{
    cmd.split(':').nth(1).unwrap_or("")
}

// Funkce `payload_without_bang`:
// Vrátí payload bez koncového vykřičníku '!', pokud tam je.
//
// Příklad volání:
// - payload_without_bang("LOGIN:alice!") -> "alice"
// - payload_without_bang("LOGIN:bob")    -> "bob"
// - payload_without_bang("PING:x!")      -> "x"
//
// Pokud payload nekončí znakem '!', vrátí jej beze změny.
// Funkce nealokuje nový String, jen vrací vhodnou část vstupního textu.
fn payload_without_bang(
    cmd: &str
) -> &str
{
    let p = payload(cmd);
    if p.ends_with('!') {
        &p[..p.len() - 1]
    } else {
        p
    }
}

// Funkce `payload_len`:
// Vrátí délku payloadu bez koncového vykřičníku.
//
// Příklad volání:
// - payload_len("LOGIN:alice!") -> 5
// - payload_len("PING:server42") -> 8
//
// Funkce vrací novou hodnotu typu číslo.
// Nevrací referenci do vstupního textu.
fn payload_len(
    cmd: &str
) -> usize
{
    payload_without_bang(cmd).len()
}

// Funkce `ensure_bang`:
// Pokud řetězec nekončí znakem '!', přidá jej na konec.
// Pokud už na konci '!' je, nic nezmění.
//
// Příklad volání:
// - po ensure_bang(&mut s), kde s == "LOGOUT:bob", bude s == "LOGOUT:bob!"
// - po ensure_bang(&mut s), kde s == "PING:x!", zůstane s == "PING:x!"
//
// Funkce původní String mění, proto musí dostat mutable referenci.
fn ensure_bang(
    cmd: &mut String
) {
    if !cmd.ends_with('!') {
        cmd.push('!');
    }
}

// Funkce `first_two`:
// Vrátí slice obsahující první dva prvky vstupního pole nebo slice.
//
// Příklad volání:
// - first_two(&[10, 20, 30, 40]) -> &[10, 20]
//
// Funkce nevrací nový vektor.
// Vrací pouze pohled na část vstupních dat.
//
// Předpoklad: vstup obsahuje alespoň dva prvky.
fn first_two(
    nums: &[i32]
) -> &[i32]
{
    &nums[..2]
}

fn main() {
    let a = String::from("LOGIN:alice!");
    let b = String::from("PING:server42");
    let mut c = String::from("LOGOUT:bob");

    println!("a = {a}");
    println!("b = {b}");
    println!("c pred = {c}");

    let name_a = command_name(&a);
    let name_b = command_name(&b);
    println!("command_name(&a) = {name_a}");
    println!("command_name(&b) = {name_b}");

    let payload_a = payload(&a);
    let payload_b = payload(&b);
    println!("payload(&a) = {payload_a}");
    println!("payload(&b) = {payload_b}");

    let clean_a = payload_without_bang(&a);
    let clean_b = payload_without_bang(&b);
    println!("payload_without_bang(&a) = {clean_a}");
    println!("payload_without_bang(&b) = {clean_b}");

    let len_a = payload_len(&a);
    let len_b = payload_len(&b);
    println!("payload_len(&a) = {len_a}");
    println!("payload_len(&b) = {len_b}");

    ensure_bang(&mut c);
    println!("c po ensure_bang = {c}");

    let values = [10, 20, 30, 40];
    println!("values = {:?}", values);

    let part = first_two(&values);
    println!("first_two(&values) = {:?}", part);

    assert_eq!(name_a, "LOGIN");
    assert_eq!(name_b, "PING");

    assert_eq!(payload_a, "alice!");
    assert_eq!(payload_b, "server42");

    assert_eq!(clean_a, "alice");
    assert_eq!(clean_b, "server42");

    assert_eq!(len_a, 5);
    assert_eq!(len_b, 8);

    assert_eq!(c, "LOGOUT:bob!");

    assert_eq!(part, &[10, 20]);
    assert_eq!(part.len(), 2);
}
main();

a = LOGIN:alice!
b = PING:server42
c pred = LOGOUT:bob
command_name(&a) = LOGIN
command_name(&b) = PING
payload(&a) = alice!
payload(&b) = server42
payload_without_bang(&a) = alice
payload_without_bang(&b) = server42
payload_len(&a) = 5
payload_len(&b) = 8
c po ensure_bang = LOGOUT:bob!
values = [10, 20, 30, 40]
first_two(&values) = [10, 20]
